# Version 3

In [ ]:
import os
import json
import torch
import gc
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# ==========================================
# 1. HARDWARE SETUP (Strict GPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != 'cuda':
    raise SystemError("GPU not detected! Enable T4 GPU in Colab Runtime.")

torch.cuda.empty_cache()
gc.collect()

153

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# 2. CUAD JSON PARSING & DATASET CREATION
# ==========================================
print("Parsing CUAD JSON...")

cuad_path = os.path.join("drive/MyDrive/Sem 2/AIG230/Project/Dataset", "CUAD_v1.json")

with open(cuad_path, "r", encoding="utf-8") as f:
    cuad_raw = json.load(f)

records = []
label_names = []

# Extract clauses and their categories from the SQuAD-formatted JSON
for doc in cuad_raw["data"]:
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            if not qa["is_impossible"]: # We only want actual extracted clauses
                question = qa["title"] if "title" in qa else qa["question"]
                # Clean up the label name
                category = question.split("related to: ")[-1].strip() if "related to: " in question else question

                if category not in label_names:
                    label_names.append(category)

                label_id = label_names.index(category)

                for ans in qa["answers"]:
                    text = ans["text"].strip()
                    if len(text) > 15: # Filter out single-word junk
                        records.append({"text": text, "label": label_id})

df = pd.DataFrame(records)
num_classes = len(label_names)
print(f"Extracted {len(df)} clauses across {num_classes} unique categories.")

# Convert to Hugging Face Dataset and split into Train/Test
full_dataset = Dataset.from_pandas(df)
train_test_split = full_dataset.train_test_split(test_size=0.15, seed=42)
dataset = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

Parsing CUAD JSON...
Extracted 12254 clauses across 41 unique categories.


In [ ]:
# ==========================================
# 3. INITIALIZE MODEL & TOKENIZER
# ==========================================
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [ ]:
# ==========================================
# 4. TOKENIZATION
# ==========================================
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256 # Cap length to save GPU VRAM
    )

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Tokenizing data...


Map:   0%|          | 0/10415 [00:00<?, ? examples/s]

Map:   0%|          | 0/1839 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 5. METRICS & TRAINING
# ==========================================
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

save_directory = "./saved_cuad_legal_bert"

training_args = TrainingArguments(
    output_dir=save_directory,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Critical for T4 GPU performance
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting GPU Training...")
trainer.train()

Starting GPU Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.863209,0.861606,0.772703,0.735326
2,0.871040,0.690257,0.794997,0.776109
3,0.695103,0.663833,0.790647,0.772418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1953, training_loss=1.017679178342414, metrics={'train_runtime': 514.2852, 'train_samples_per_second': 60.754, 'train_steps_per_second': 3.798, 'total_flos': 2915311504857204.0, 'train_loss': 1.017679178342414, 'epoch': 3.0})

In [ ]:
# ==========================================
# 6. EXPLICITLY SAVE THE MODEL AND TOKENIZER
# ==========================================
print(f"Saving finalized model and tokenizer to {save_directory}...")
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

# Clear VRAM so we can demonstrate loading
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

Saving finalized model and tokenizer to ./saved_cuad_legal_bert...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4936

In [ ]:
# ==========================================
# 7. LOAD AND REUSE THE SAVED MODEL
# ==========================================
print("\n--- Testing Reloaded Model ---")

# Load directly from the local directory we just saved to
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory).to(device)
loaded_model.eval()

# A sample clause to test
sample_text = "The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties."

# Run inference
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=256).to(device)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction_id = torch.argmax(outputs.logits, dim=-1).item()

# Use the model's built-in id2label mapping (which we configured during initialization)
predicted_label = loaded_model.config.id2label[prediction_id]

print(f"Text: '{sample_text}'")
print(f"Predicted Clause Type: **{predicted_label}**")


--- Testing Reloaded Model ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: 'The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties.'
Predicted Clause Type: **Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector? **


In [ ]:
import shutil
source = '/content/saved_cuad_legal_bert'

destination = '/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3'

shutil.move(source, destination)

'/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3/saved_cuad_legal_bert'